### 2.2 Tri par le texte
On parcourt tous les paragraphes du texte et on compte combien de mots-cles **distincts** apparaissent. Si le seuil est atteint, la seance est retenue.

> **Correction methodologique** : on compte les racines distinctes qui matchent (et non le nombre total d'occurrences), pour eviter qu'un seul mot repete plusieurs fois dans le texte atteigne artificiellement le seuil.

## 1. Imports et configuration

In [1]:
from config import *
from lxml import etree
import os, re
from tqdm import tqdm
from shutil import copyfile

# Namespace XML de l'Assemblée Nationale
NAMESPACES = {'ns': 'http://schemas.assemblee-nationale.fr/referentiel'}

print("Configuration chargée.")

Configuration chargée.


## 2. Fonctions de tri

### 2.1 Tri par le sommaire
On parse le sommaire XML de chaque séance pour y chercher les mots-clés VSS dans les titres de niveau 1 et 2.

In [2]:
def tri_titres(fichier, liste):
    """
    Teste si le sommaire d'un fichier XML contient un des mots-clés de la liste.

    Args:
        fichier (str): chemin complet vers le fichier XML
        liste (list): liste de racines de mots à chercher

    Returns:
        bool: True si au moins un titre du sommaire contient un mot-clé
    """
    try:
        tree = etree.parse(fichier)
        root = tree.getroot()
        for chaine in liste:
            chaine_lower = chaine.lower()
            # Titres de niveau 1
            for titre in root.xpath('//ns:sommaire1//ns:intitule', namespaces=NAMESPACES):
                if titre.text and chaine_lower in titre.text.lower():
                    return True
            # Titres de niveau 2
            for titre in root.xpath('//ns:sommaire2//ns:intitule', namespaces=NAMESPACES):
                if titre.text and chaine_lower in titre.text.lower():
                    return True
        return False
    except Exception:
        return False

### 2.2 Tri par le texte
On parcourt tous les paragraphes du texte et on compte combien de mots-clés différents apparaissent. Si le seuil est atteint, la séance est retenue.

In [3]:
def tri_textes(fichier, liste, seuil):
    """
    Compte le nombre de mots-cles VSS DISTINCTS dans le texte d'un fichier XML.

    Correction : on compte les racines distinctes qui matchent (set),
    et non le nombre total d'occurrences, pour eviter qu'un seul mot
    repete n fois atteigne le seuil.

    Args:
        fichier (str): chemin complet vers le fichier XML
        liste (list): liste de racines de mots a chercher
        seuil (int): nombre minimum de mots-cles distincts pour retenir le fichier

    Returns:
        bool: True si le nombre de mots-cles distincts atteint le seuil
    """
    try:
        tree = etree.parse(fichier)
        root = tree.getroot()
        mots_trouves = set()
        for para in root.xpath('//ns:paragraphe', namespaces=NAMESPACES):
            text_nodes = para.xpath('.//ns:texte//text()', namespaces=NAMESPACES)
            texte_complet = " ".join(text_nodes).lower()
            for chaine in liste:
                pattern = rf'\b{re.escape(chaine.lower())}'
                if re.search(pattern, texte_complet):
                    mots_trouves.add(chaine)
        return len(mots_trouves) >= seuil
    except Exception:
        return False

## 3. Exécution du tri

On parcourt les 3 législatures. Pour chaque fichier XML, on applique les deux filtres. Si l'un des deux est positif, le fichier est ajouté à la liste des séances pertinentes.


In [4]:
fichier_cache = os.path.join(DOSSIER_DATAFRAMES, "liste_seances_pertinentes.txt")

# Suppression du cache pour forcer le recalcul
# (necessaire apres modification du pipeline de filtrage)
if os.path.exists(fichier_cache):
    os.remove(fichier_cache)
    print("Cache supprime, recalcul force.")

liste_pertinente = []

for nom_leg, chemin_dossier in DOSSIERS_LEGISLATURES.items():
    if not os.path.exists(chemin_dossier):
        print(f"Dossier introuvable : {chemin_dossier}")
        continue

    fichiers_xml = [f for f in os.listdir(chemin_dossier) if f.endswith('.xml')]
    print(f"\nLegislature {nom_leg} : {len(fichiers_xml)} fichiers a analyser")

    for fichier in tqdm(fichiers_xml, desc=f"Tri {nom_leg}"):
        chemin_complet = os.path.join(chemin_dossier, fichier)

        if tri_titres(chemin_complet, A_TESTER):
            liste_pertinente.append(fichier)
            continue

        if tri_textes(chemin_complet, A_TESTER, SEUIL_TRI):
            liste_pertinente.append(fichier)

# Suppression des doublons
liste_pertinente = list(set(liste_pertinente))

# Sauvegarde du cache
with open(fichier_cache, "w") as f:
    f.write("\n".join(liste_pertinente))

print(f"\nTri termine : {len(liste_pertinente)} seances retenues.")
print(f"Cache sauvegarde dans {fichier_cache}")

Cache supprime, recalcul force.

Legislature XV : 1563 fichiers a analyser


Tri XV:   0%|          | 0/1563 [00:00<?, ?it/s]

Tri XV:   0%|          | 1/1563 [00:00<04:47,  5.44it/s]

Tri XV:   0%|          | 2/1563 [00:00<05:27,  4.77it/s]

Tri XV:   0%|          | 3/1563 [00:00<04:24,  5.89it/s]

...

Tri XVII:  99%|█████████▉| 363/365 [00:51<00:00,  5.80it/s]

Tri XVII: 100%|█████████▉| 364/365 [00:51<00:00,  6.47it/s]

Tri XVII: 100%|██████████| 365/365 [00:51<00:00,  7.17it/s]

Tri XVII: 100%|██████████| 365/365 [00:51<00:00,  7.06it/s]


Tri termine : 1466 seances retenues.
Cache sauvegarde dans /home/onyxia/work/projet_eco_socio/dataframes/liste_seances_pertinentes.txt


## 4. Copie des séances sélectionnées

On copie les fichiers retenus dans les dossiers `sorted/` pour les séparer des données brutes. Cela permet de ne travailler que sur les fichiers pertinents dans les notebooks suivants.

In [5]:
# ==========================================================================
# COPIE DES FICHIERS
# ==========================================================================

for nom_leg, chemin_source in DOSSIERS_LEGISLATURES.items():
    cle_sorted = nom_leg.lower()
    if cle_sorted not in DOSSIER_SORTED:
        continue

    chemin_dest = DOSSIER_SORTED[cle_sorted]
    os.makedirs(chemin_dest, exist_ok=True)

    if not os.path.exists(chemin_source):
        continue

    nb_copies = 0
    for fichier in os.listdir(chemin_source):
        if fichier in liste_pertinente:
            dest = os.path.join(chemin_dest, fichier)
            if not os.path.exists(dest):
                copyfile(os.path.join(chemin_source, fichier), dest)
                nb_copies += 1

    print(f"{nom_leg} : {nb_copies} nouveaux fichiers copiés dans {chemin_dest}")

print("\nCopie terminée.")

XV : 0 nouveaux fichiers copiés dans /home/onyxia/work/projet_eco_socio/sorted/xv
XVI : 0 nouveaux fichiers copiés dans /home/onyxia/work/projet_eco_socio/sorted/xvi
XVII : 0 nouveaux fichiers copiés dans /home/onyxia/work/projet_eco_socio/sorted/xvii

Copie terminée.


## 5. Vérification

In [6]:
# Vérification du contenu des dossiers sorted/
for leg, chemin in DOSSIER_SORTED.items():
    if os.path.exists(chemin):
        nb = len([f for f in os.listdir(chemin) if f.endswith('.xml')])
        print(f"  {leg.upper()} : {nb} fichiers XML")
    else:
        print(f"  {leg.upper()} : dossier non trouvé")

print(f"\nTotal de séances retenues : {len(liste_pertinente)}")

  XV : 1219 fichiers XML
  XVI : 463 fichiers XML
  XVII : 281 fichiers XML

Total de séances retenues : 1466
